# DeepMIMO City Pool — Cross-Layout Test

This notebook is a notebook-friendly conversion of `rmbench_city.py`. It preserves the original configuration, candidate city list, geometry-based features, build logic, and role assignment.

**Important:** building the city pool can trigger large downloads (potentially several GB). Run the build section only when ready. The notebook expects `rmbench_core.py`, `rmbench_dmfix.py`, and the project's dependencies to be available in the working directory/environment.

## 1. Imports and configuration

In [1]:
import os, sys, re, json, math, time, hashlib
import numpy as np

# Make project modules importable from the current working directory.
sys.path.insert(0, os.getcwd())
import rmbench_core as rc
import rmbench_dmfix  # carrier-frequency fix; must precede any dataset load

CITY_CFG = dict(
    rc.DEFAULT_CFG,
    cache_dir="rmbench_cache_city",
    max_points=12000,
    min_unit_points=1000,
    n_test=500,
    densities=(30, 100, 300),
    n_rep=5,
    n_jobs=8,
)
MANIFEST, PLAN = "city_manifest.csv", "results_city/prereg_city.json"
ENV_COLS = [
    "log1p_n_edges",
    "log1p_blocked_len",
    "log1p_max_obstruction",
    "cover_30m",
    "log1p_d_nearest",
]
RES, PAD, R_COVER = 2.0, 40.0, 30.0
MIN_ROOF = 2.0
FEAT_VERSION = "raster-v2"

_US = ["newyork", "losangeles", "chicago", "houston", "phoenix", "philadelphia", "miami", "sandiego", "dallas", "sanfrancisco", "austin",
       "santaclara", "fortworth", "columbus", "charlotte", "indianapolis", "sanfrancisco", "seattle", "denver", "oklahoma"]
_REST = {21: "taito_city", 22: "rome", 23: "beijing", 24: "cairo", 25: "instanbul", 26: "dubai", 27: "rio_de_janeiro", 28: "mumbai", 29: "centro",
         30: "singapore", 31: "barcelona", 32: "sydney", 33: "hong_kong", 34: "amsterdam", 35: "san_francisco", 36: "bangkok", 37: "seoul",
         38: "toronto", 39: "jerusalem", 40: "prague", 41: "kyoto", 42: "san_nicolas", 43: "cape_town", 44: "lisboa", 45: "stockholm",
         46: "la_habana", 47: "chicago", 48: "gurbchen", 49: "new_delhi", 50: "edinburgh", 51: "firenze", 52: "marrakesh", 53: "sankt-peterburg",
         54: "north_jakarta", 55: "madrid", 56: "montreal", 57: "taipei", 58: "sumida_city", 59: "roma", 60: "toronto", 61: "hatsukaichi",
         62: "santiago", 63: "athens", 64: "new_york", 65: "granada", 66: "bruxelles", 67: "fujiyoshida", 68: "reykjavik", 69: "warszawa",
         70: "stockholm", 71: "helsinki", 72: "capetown", 73: "casablanca", 74: "chiyoda", 75: "dongcheng", 76: "houston", 77: "melbourne",
         78: "newtaipei", 79: "parow", 80: "philadelphia", 81: "seoul", 82: "shibuya", 83: "tempe", 84: "baoding", 85: "fuyang", 86: "ankara",
         87: "jinan", 88: "tongshan", 89: "nairobi", 90: "nanjing", 91: "xiangyang", 93: "rangoon", 94: "monterrey", 95: "delhi", 96: "osaka",
         97: "chennai", 98: "guangzhou", 99: "shanghai", 101: "jakarta", 102: "huanggang"}
CANDIDATES = [f"city_{i}_{n}_3p5" for i, n in enumerate(_US)] + [f"city_{i}_{n}_3p5" for i, n in sorted(_REST.items())]

_ALIAS = {
    "roma": "rome", "new_york": "newyork", "san_francisco": "sanfrancisco",
    "taito_city": "tokyo", "chiyoda": "tokyo", "sumida_city": "tokyo",
    "shibuya": "tokyo", "north_jakarta": "jakarta", "new_delhi": "delhi",
    "cape_town": "capetown", "newtaipei": "taipei",
}
print(f"Configured {len(CANDIDATES)} candidate scenarios.")


Configured 100 candidate scenarios.


## 2. City grouping and role assignment

In [2]:
def parse_slug(s):
    m = re.match(r"city_(\d+)_(.+)_3p5$", s)
    return int(m.group(1)), _ALIAS.get(m.group(2), m.group(2))


def groups(slugs):
    """Group aliases so layouts from the same city stay together."""
    grouped = {}
    for slug in slugs:
        index, name = parse_slug(slug)
        grouped.setdefault(name, []).append((index, slug))
    return sorted(
        ((name, [slug for _, slug in sorted(items)])
         for name, items in grouped.items()),
        key=lambda item: min(parse_slug(slug)[0] for slug in item[1]),
    )


def assign_roles(eligible):
    """Assign roles deterministically by eligible group position; no randomness."""
    role = {}
    for position, (_, slugs) in enumerate(groups(eligible)):
        r = (
            "source" if position % 8 <= 3
            else "val" if position % 8 == 4
            else "dev" if position % 8 == 5
            else "confirm"
        )
        for slug in slugs:
            role[slug] = r
    return role


## 3. Geometry rasterization and environment features

In [3]:
def _normal(v):
    """Newell normal; robust to collinear first vertices and concave outlines."""
    v = np.asarray(v, float)
    normal = np.cross(v, np.roll(v, -1, axis=0)).sum(0)
    norm = np.linalg.norm(normal)
    return normal / norm if norm > 1e-12 else np.zeros(3)


def rasterize(scene, x0, y0, nx, ny, res=RES, label="buildings", min_roof=MIN_ROOF):
    """Rasterize horizontal building faces into occupancy and roof-height grids."""
    occ = np.zeros((ny, nx), bool)
    hgt = np.zeros((ny, nx))
    for obj in scene.get_objects(label):
        for face in obj.faces:
            vertices = np.asarray(face.vertices, float)
            if len(vertices) < 3 or abs(_normal(vertices)[2]) < 0.7:
                continue
            z = float(vertices[:, 2].mean())
            if z < min_roof:
                continue
            for tri in face.triangular_faces:
                T = np.asarray(tri, float)[:, :2]
                ix0 = max(int(np.floor((T[:, 0].min() - x0) / res)), 0)
                ix1 = min(int(np.ceil((T[:, 0].max() - x0) / res)), nx - 1)
                iy0 = max(int(np.floor((T[:, 1].min() - y0) / res)), 0)
                iy1 = min(int(np.ceil((T[:, 1].max() - y0) / res)), ny - 1)
                if ix1 < ix0 or iy1 < iy0:
                    continue
                gx, gy = np.meshgrid(
                    x0 + (np.arange(ix0, ix1 + 1) + 0.5) * res,
                    y0 + (np.arange(iy0, iy1 + 1) + 0.5) * res,
                )
                (ax, ay), (bx, by), (cx, cy) = T
                den = (by - cy) * (ax - cx) + (cx - bx) * (ay - cy)
                if abs(den) < 1e-12:
                    continue
                l1 = ((by - cy) * (gx - cx) + (cx - bx) * (gy - cy)) / den
                l2 = ((cy - ay) * (gx - cx) + (ax - cx) * (gy - cy)) / den
                l3 = 1 - l1 - l2
                mask = (l1 >= 0) & (l2 >= 0) & (l3 >= 0)
                sub = (slice(iy0, iy1 + 1), slice(ix0, ix1 + 1))
                occ[sub] |= mask
                hgt[sub] = np.where(mask, np.maximum(hgt[sub], z), hgt[sub])
    return occ, hgt


def raster_for(scene, pos3, tx3, res=RES, pad=PAD):
    """Build a raster covering receiver and transmitter positions plus padding."""
    points = np.vstack([
        np.asarray(pos3, float)[:, :2],
        np.asarray(tx3, float)[None, :2],
    ])
    x0, y0 = points.min(0) - pad
    x1, y1 = points.max(0) + pad
    nx, ny = int(np.ceil((x1 - x0) / res)), int(np.ceil((y1 - y0) / res))
    occ, hgt = rasterize(scene, x0, y0, nx, ny, res)
    return {"occ": occ, "hgt": hgt, "x0": x0, "y0": y0, "res": res}


def _cells(R, P):
    ix = np.clip(np.floor((P[..., 0] - R["x0"]) / R["res"]).astype(int), 0, R["occ"].shape[1] - 1)
    iy = np.clip(np.floor((P[..., 1] - R["y0"]) / R["res"]).astype(int), 0, R["occ"].shape[0] - 1)
    return iy, ix


def on_building(R, P):
    iy, ix = _cells(R, np.asarray(P, float))
    return R["occ"][iy, ix]


def env_features(R, rx3, tx3, chunk=1500, r_cover=R_COVER):
    """Return five label-free map features per receiver; no channel outputs are used."""
    from scipy import ndimage
    rx3 = np.asarray(rx3, float)
    tx3 = np.asarray(tx3, float).reshape(3)
    occ, hgt, res = R["occ"], R["hgt"], R["res"]
    dist = ndimage.distance_transform_edt(~occ) * res
    cov = ndimage.uniform_filter(
        occ.astype(float), size=int(2 * r_cover / res) + 1, mode="constant"
    )
    L = np.hypot(rx3[:, 0] - tx3[0], rx3[:, 1] - tx3[1])
    K = int(np.ceil(L.max() / res)) + 2
    t = np.linspace(0.0, 1.0, K)
    out = np.zeros((len(rx3), 5))
    for start in range(0, len(rx3), chunk):
        P = rx3[start:start + chunk]
        pts = tx3[None, None, :2] + (
            P[:, None, :2] - tx3[None, None, :2]
        ) * t[None, :, None]
        iy, ix = _cells(R, pts)
        z = tx3[2] + (P[:, 2:3] - tx3[2]) * t[None, :]
        occupied = occ[iy, ix]
        heights = hgt[iy, ix]
        own = np.cumprod(occupied, axis=1).astype(bool)
        obstruction = np.where(
            own | ~occupied, 0.0, np.maximum(heights - z, 0.0)
        )
        blocked = obstruction > 0
        edges = (blocked[:, 1:] & ~blocked[:, :-1]).sum(1) + blocked[:, 0]
        riy, rix = _cells(R, P[:, :2])
        out[start:start + chunk] = np.column_stack([
            np.log1p(edges),
            np.log1p(blocked.mean(1) * L[start:start + chunk]),
            np.log1p(obstruction.max(1)),
            cov[riy, rix],
            np.log1p(dist[riy, rix]),
        ])
    return out


## 4. Build units for one city scenario

In [4]:
def build_city(slug, cfg=CITY_CFG):
    """Build and save one unit per transmitter for a scenario."""
    names, drops, n_buildings, raster = [], [], 0, None
    inside_kept_values, inside_nopath_values = [], []

    for k, ds in enumerate(rc._dm_datasets(slug, cfg)):
        pos3 = np.asarray(ds.rx_pos, float)
        tx3 = np.asarray(ds.tx_pos, float).reshape(-1)[:3]
        raw = {
            "pos3": pos3,
            "tx3": tx3,
            "power_paths_dbm": rc._path_power_dbm(ds),
            "los": np.asarray(ds.los),
            "freq_ghz": rc._carrier_ghz(ds, slug),
            "name": f"dm|{slug}|{k}",
            "scenario": slug,
            "tx": k,
        }
        unit = rc.preprocess_unit(raw, cfg)
        if unit is None:
            drops.append(k)
            continue

        distances = np.linalg.norm(pos3 - tx3, axis=1)
        power = rc.sum_path_power_dbm(raw["power_paths_dbm"])
        los = (np.asarray(raw["los"]) == 1).astype(float)
        y = power + rc.calibrate_power_offset(
            distances, power, los, raw["freq_ghz"], cfg["ptx_dbm"]
        )
        keep = np.isfinite(y) & (y > rc.noise_floor_dbm(cfg))
        assert keep.sum() == len(unit["y"]), "keep mask does not reproduce preprocess_unit"

        if raster is None:
            scene = getattr(ds, "scene", None)
            n_buildings = len(scene.get_objects("buildings")) if scene is not None else 0
            raster = raster_for(scene, pos3, tx3) if n_buildings else None

        side = env_features(raster, pos3[keep], tx3) if raster is not None else np.zeros(
            (int(keep.sum()), len(ENV_COLS))
        )
        if raster is not None:
            no_path = np.asarray(raw["los"]) == -1
            inside_kept_values.append(float(on_building(raster, pos3[keep, :2][::5]).mean()))
            if no_path.sum():
                inside_nopath_values.append(float(on_building(raster, pos3[no_path, :2][::5]).mean()))
            else:
                inside_nopath_values.append(float("nan"))

        unit.update(
            layout=slug,
            parent=slug,
            tx_known=1,
            side=side.astype(np.float32),
            has_scene=int(raster is not None),
            n_buildings=n_buildings,
        )
        rc.save_unit(unit, cfg["cache_dir"])
        names.append(unit["name"])

    inside_kept = float(np.mean(inside_kept_values)) if inside_kept_values else float("nan")
    inside_nopath = (
        float(np.nanmean(inside_nopath_values))
        if inside_nopath_values and not all(v != v for v in inside_nopath_values)
        else float("nan")
    )
    info = {
        "slug": slug,
        "n_units": len(names),
        "n_dropped_units": len(drops),
        "n_buildings": n_buildings,
        "inside_kept": inside_kept,
        "inside_nopath": inside_nopath,
    }
    return names, info


def eligible_row(row):
    inside_kept = row.get("inside_kept", float("nan"))
    inside_nopath = row.get("inside_nopath", float("nan"))
    return bool(
        row["n_units"] >= 1
        and row["n_buildings"] >= 10
        and inside_kept == inside_kept
        and inside_nopath == inside_nopath
        and inside_kept <= 0.10
        and (inside_nopath - inside_kept) >= 0.25
    )


def build_with_retry(slug, cfg=CITY_CFG, waits=(45, 120, 300), pause=4.0):
    """Retry transient missing-scenario/HTTP 429 download failures."""
    import deepmimo as dm
    for attempt, wait_seconds in enumerate((0,) + tuple(waits)):
        if wait_seconds:
            print(
                f"  [{slug}] download refused, waiting {wait_seconds}s "
                f"(attempt {attempt + 1}/{len(waits) + 1})"
            )
            time.sleep(wait_seconds)
        try:
            if not os.path.isdir(os.path.join(dm.get_scenarios_dir(), slug)):
                time.sleep(pause)
            return build_city(slug, cfg)
        except Exception as exc:
            if attempt == len(waits) or (
                "not found" not in repr(exc).lower()
                and "429" not in repr(exc)
            ):
                raise


## 5. Build the city manifest

Choose how many eligible city groups to attempt. `first=24` matches the script's documented example. This downloads/builds scenarios and can take a long time.

Set `FIRST_GROUPS = None` to process all candidates (potentially several GB of downloads).

In [5]:
import pandas as pd

def cmd_build(first=None, cfg=CITY_CFG):
    todo = CANDIDATES if first is None else [
        slug for _, slugs in groups(CANDIDATES)[:first] for slug in slugs
    ]
    all_rows = pd.read_csv(MANIFEST).to_dict("records") if os.path.exists(MANIFEST) else []
    rows = [
        row for row in all_rows
        if row.get("feat_version") == FEAT_VERSION
        and not (isinstance(row.get("error"), str) and row["error"])
    ]
    done = {row["slug"] for row in rows}
    if len(all_rows) != len(rows):
        print(
            f"[city] {len(all_rows) - len(rows)} manifest rows are old or failed "
            "and will be rebuilt"
        )

    for slug in todo:
        if slug in done:
            continue
        start = time.time()
        try:
            _, info = build_with_retry(slug, cfg)
        except Exception as exc:
            info = {
                "slug": slug,
                "n_units": 0,
                "n_dropped_units": -1,
                "n_buildings": 0,
                "inside_kept": float("nan"),
                "inside_nopath": float("nan"),
                "error": repr(exc)[:160],
            }
        info["eligible"] = eligible_row(info)
        info["feat_version"] = FEAT_VERSION
        rows.append(info)
        pd.DataFrame(rows).to_csv(MANIFEST, index=False)
        print(
            f"{slug}: units={info['n_units']} dropped={info['n_dropped_units']} "
            f"bldgs={info['n_buildings']} on_bldg(signal)={info['inside_kept']:.2f} "
            f"on_bldg(no path)={info['inside_nopath']:.2f} "
            f"eligible={info['eligible']} ({time.time() - start:.0f}s) "
            f"{info.get('error', '')}"
        )

    manifest_df = pd.read_csv(MANIFEST)
    print(f"\n{int(manifest_df.eligible.sum())}/{len(manifest_df)} eligible -> {MANIFEST}")
    return manifest_df


# Run this cell only when ready to download/build:
FIRST_GROUPS = 24
# FIRST_GROUPS = None  # uncomment to process every candidate
# manifest_df = cmd_build(FIRST_GROUPS)


## 6. Assign and optionally freeze roles

Review the role table before freezing it. Once `results_city/prereg_city.json` exists, roles are read from that frozen plan; newly eligible scenarios receive `confirm2`.

In [6]:
def cmd_roles(freeze=False):
    df = pd.read_csv(MANIFEST)
    eligible = df[df.eligible].slug.tolist()

    if os.path.exists(PLAN):
        with open(PLAN, "r", encoding="utf-8") as f:
            frozen_plan = json.load(f)
        frozen_roles = frozen_plan["plan"]["roles"]
        role = {slug: frozen_roles.get(slug, "confirm2") for slug in eligible}
        print(
            f"Plan already frozen ({frozen_plan['sha256'][:16]}): "
            f"{len(frozen_roles)} roles kept, "
            f"{sum(1 for slug in eligible if slug not in frozen_roles)} "
            "newly eligible -> confirm2"
        )
        if freeze:
            print("(freeze ignored: the plan is already frozen)")
    else:
        role = assign_roles(eligible)

    df["role"] = df.slug.map(role).fillna("ineligible")
    df.to_csv(MANIFEST, index=False)
    summary = df.groupby("role").slug.agg([
        "size",
        lambda x: ", ".join(x.str.extract(r"city_(\d+)_")[0].head(12)),
    ])
    display(summary)

    if os.path.exists(PLAN):
        return df

    plan = {
        "cfg": {key: value for key, value in CITY_CFG.items() if key != "cache_dir"},
        "env_cols": ENV_COLS,
        "roles": role,
        "rule": "p%8: 0-3 source, 4 val, 5 dev, 6-7 confirm; groups by alias",
    }
    digest = hashlib.sha256(
        json.dumps(plan, sort_keys=True, default=str).encode()
    ).hexdigest()

    if freeze:
        os.makedirs(os.path.dirname(PLAN), exist_ok=True)
        with open(PLAN, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "sha256": digest,
                    "plan": plan,
                    "frozen_at": time.strftime("%Y-%m-%d %H:%M:%S"),
                },
                f,
                indent=1,
            )
        print("Frozen", digest[:16], "->", PLAN)
    else:
        print("(not frozen; add freeze=True after reviewing the table)", digest[:16])
    return df


# First review roles without freezing:
# roles_df = cmd_roles(freeze=False)
# Freeze only after reviewing:
# roles_df = cmd_roles(freeze=True)


## 7. Run offline (no downloads)

Uses the already-built pool. Run the next cell, then the one after it.

In [7]:
# OFFLINE: reads the existing manifest/cache only. cmd_build (the downloader) is never called here.
assert os.path.exists(MANIFEST), f"{MANIFEST} not found in {os.getcwd()} - run this notebook from the project folder"
manifest_df = pd.read_csv(MANIFEST)
print(len(manifest_df), "scenarios in manifest,", int(manifest_df.eligible.sum()), "eligible")
roles_df = cmd_roles(freeze=False)   # review the table; freezing is a separate, deliberate step


92 scenarios in manifest, 53 eligible
Plan already frozen (6a9c40aaf360a151): 53 roles kept, 0 newly eligible -> confirm2


,size,<lambda_0>
role,,
confirm,10,"6, 7, 14, 15, 27, 31, 54, 56, 73, 75"
dev,7,"5, 80, 13, 26, 53, 68, 87"
ineligible,39,"64, 47, 17, 22, 23, 28, 29, 30, 32, 36, 37, 38"
source,30,"0, 1, 2, 3, 76, 8, 9, 16, 35, 10, 11, 18"
val,6,"4, 12, 25, 52, 67, 86"


In [8]:
# Start the work: runs run_city_gnn.py in DEV mode through uv (exploratory only, not confirmatory). No downloads.
import subprocess
NO_GNN = False                               # True = classical methods only (fast). Set False to include GCN/GAT.
N_JOBS = max(1, (os.cpu_count() or 2) // 2)
cmd = ["uv", "run", "run_city_gnn.py", "dev", "--n-jobs", str(N_JOBS)] + (["--no-gnn"] if NO_GNN else [])
print(" ".join(cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
print("exit code:", proc.wait())


uv run run_city_gnn.py dev --n-jobs 8
[city_gnn_dev] role=dev: 12 units in 7 cities, densities (30, 100, 300), reps 5, 13 jobs: ['trend', 'idw', 'krig_global', 'kgnn', 'rk_spline', 'rk_gbm', 'gcn_knn_side', 'gat_del_side', 'rk_side', 'gcn_knn', 'gcn_del', 'gat_knn', 'gat_del']
[run_tasks6] 180 tasks, 124 done, 56 to run (n_jobs=8) -> results_city/city_gnn_dev.jsonl
  25/56 209s failed jobs: 0
  50/56 375s failed jobs: 0
  56/56 436s failed jobs: 0
[check_run] results_city/city_gnn_dev.jsonl: 2340 rows, 0 failed (0.00%)

RMSE (dB), mean over cities (units and replicates averaged within a city):
density         30     100    300
name                             
gat_del       5.878  4.097  3.179
gat_del_side  4.366  3.302  2.637
gat_knn       6.065  4.024  3.089
gcn_del       5.976  4.233  3.213
gcn_knn       6.057  4.119  3.138
gcn_knn_side  4.486  3.362  2.780
idw           5.311  3.739  2.831
kgnn          5.086  3.542  2.603
krig_global   5.252  3.614  2.670
rk_gbm        4.125  2.92